# 05 — Component ablation on the official CEC 2017 suite

Produces `data/ablation_official_cec2017_30D.json`, the data behind **Table 7** of the
manuscript.

## What this does

Four variants are compared on ten CEC 2017 functions at 30 dimensions, 30 independent runs
each (1,200 runs total):

| Variant | Change from the published algorithm |
|---|---|
| `QSO-Full` | none — Eq. (1)–(10) as specified |
| `QSO-Static` | θ fixed at the midpoint 0.5 instead of declining from 0.7 to 0.3 |
| `QSO-NoReset` | signal decay of Eq. (10) disabled (τ set beyond the run length) |
| `QSO-NoCentroid` | centroid term removed from Eq. (8); attraction to the incumbent best only |

`qso()` is reproduced verbatim below. The NoCentroid variant is created by substituting
`exploitation_phase` in the function's global namespace rather than by editing the
algorithm, so `QSO-Full` is provably the published implementation.

## Why the official suite

An earlier version of this ablation used CEC 2017 functions with shift vectors but **without**
the competition rotation matrices, and produced the opposite ordering: the centroid term
appeared to contribute nothing. Rotation removes separability, and centroid attraction
supplies the coherent population movement that rotated basins require. The reversal is
reported in Section 5.2 of the manuscript. This notebook therefore uses `opfunu` 1.0.1,
which loads the official competition data files.

## Runtime

Roughly 10–30 minutes depending on core count. Checkpoints are written after every
variant-function cell, so a disconnect costs at most one cell.

In [ ]:
# === 1. Setup ===
from google.colab import drive
drive.mount('/content/drive')

!pip -q install opfunu==1.0.1

import os, json, time, types
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import rankdata, wilcoxon
from scipy.special import gamma

BASE = '/content/drive/MyDrive/QSO_Research'
OUT  = f'{BASE}/ablation_official'
os.makedirs(OUT, exist_ok=True)
RESULTS = f'{OUT}/ablation_official_cec2017_30D.json'

FUNCS  = [1, 3, 4, 5, 6, 9, 10, 21, 23, 25]   # unimodal, multimodal, hybrid, composition
SEEDS  = list(range(42, 72))                   # 30 runs, as in the manuscript
DIM    = 30
POP    = 30
ITERS  = 500

print('output:', RESULTS)
print('exists :', os.path.exists(RESULTS))

In [ ]:
# === 2. QSO — verbatim from src/qso.py ===

import numpy as np

def levy_flight(n, d, alpha=1.25):
    from scipy.special import gamma
    sigma_u = (
        gamma(1 + alpha) * np.sin(np.pi * alpha / 2) /
        (gamma((1 + alpha) / 2) * alpha * 2**((alpha-1)/2))
    ) ** (1/alpha)
    u = np.random.normal(0, sigma_u, (n, d))
    v = np.random.normal(0, 1.0, (n, d))
    return u / (np.abs(v) ** (1/alpha))

def clip_to_bounds(x, lb, ub):
    repair_lb = lb + np.random.rand(*x.shape) * (ub - lb) * 0.1
    repair_ub = ub - np.random.rand(*x.shape) * (ub - lb) * 0.1
    x = np.where(x < lb, repair_lb, x)
    x = np.where(x > ub, repair_ub, x)
    return x

def compute_ai_concentration(fitness_values, f_best, f_worst):
    epsilon = 1e-10
    if (f_worst - f_best) < epsilon:
        return 0.5
    phi = (f_worst - fitness_values) / (f_worst - f_best + epsilon)
    return float(np.clip(np.mean(phi), 0.0, 1.0))

def adaptive_threshold(t, max_iter, theta_min=0.3, theta_max=0.7):
    return float(theta_max - (theta_max - theta_min)
                 * (t / max_iter))

def exploration_phase(X, lb, ub, alpha=1.25):
    n, d = X.shape
    r1 = np.random.rand(n, d)
    r2 = np.random.rand(n, d)
    X_rand = X[np.random.randint(0, n, size=n)]
    L = levy_flight(n, d, alpha)
    scale = (ub - lb) * 0.01
    L_scaled = np.clip(L * scale, -0.5*(ub-lb), 0.5*(ub-lb))
    X_new = X + r1*(X_rand - X) + r2*L_scaled
    return clip_to_bounds(X_new, lb, ub)

def exploitation_phase(X, X_best, lb, ub):
    n, d = X.shape
    r3 = np.random.rand(n, d)
    r4 = np.random.rand(n, d)
    X_colony = np.mean(X, axis=0)
    X_new = (X
             + r3 * (X_best   - X)
             + r4 * (X_colony - X))
    return clip_to_bounds(X_new, lb, ub)

def apply_ai_decay(C, lambda_=0.05):
    return float(np.clip(C * np.exp(-lambda_), 0.0, 1.0))

def qso(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        theta_min=0.3, theta_max=0.7,
        lambda_=0.05, tau=10, alpha=1.25,
        seed=42, verbose=False):
    np.random.seed(seed)
    lb = np.full(dim, lb) if np.isscalar(lb) else np.array(lb)
    ub = np.full(dim, ub) if np.isscalar(ub) else np.array(ub)
    X = np.random.uniform(lb, ub, (pop_size, dim))
    fitness = np.array([func(X[i]) for i in range(pop_size)])
    best_idx      = np.argmin(fitness)
    best_fitness  = fitness[best_idx]
    best_position = X[best_idx].copy()
    theta_t = adaptive_threshold(
                  0, max_iter, theta_min, theta_max)
    C = compute_ai_concentration(
            fitness, best_fitness, np.max(fitness))
    convergence    = [best_fitness]
    diversity      = [np.mean(np.std(X, axis=0))]
    quorum_history = [C]
    phase_history  = [1 if C >= theta_t else 0]
    theta_history  = [theta_t]
    no_improve_count = 0
    for t in range(max_iter):
        theta_t = adaptive_threshold(
                      t, max_iter, theta_min, theta_max)
        if C >= theta_t:
            X     = exploitation_phase(
                        X, best_position, lb, ub)
            phase = 1
        else:
            X     = exploration_phase(X, lb, ub, alpha)
            phase = 0
        fitness = np.array([func(X[i]) for i in range(pop_size)])
        worst_idx = np.argmax(fitness)
        if fitness[worst_idx] > best_fitness:
            X[worst_idx]       = best_position.copy()
            fitness[worst_idx] = best_fitness
        current_best_idx     = np.argmin(fitness)
        current_best_fitness = fitness[current_best_idx]
        if current_best_fitness < best_fitness:
            best_fitness     = current_best_fitness
            best_position    = X[current_best_idx].copy()
            no_improve_count = 0
        else:
            no_improve_count += 1
        C = compute_ai_concentration(
                fitness, best_fitness, np.max(fitness))
        if no_improve_count >= tau:
            C = apply_ai_decay(C, lambda_)
            no_improve_count = 0
        convergence.append(best_fitness)
        diversity.append(np.mean(np.std(X, axis=0)))
        quorum_history.append(C)
        phase_history.append(phase)
        theta_history.append(theta_t)
        if verbose and (t+1) % 100 == 0:
            print(f"Iter {t+1}/{max_iter} | "
                  f"Best: {best_fitness:.6e} | "
                  f"C: {C:.3f} | theta: {theta_t:.3f}")
    return (best_fitness, best_position, convergence,
            diversity, quorum_history, phase_history,
            theta_history)


In [ ]:
# === 3. Ablation variants ===

def exploitation_phase_no_centroid(X, X_best, lb, ub):
    """QSO-NoCentroid: Eq. (8) with the centroid term removed."""
    n, d = X.shape
    r3 = np.random.rand(n, d)
    return clip_to_bounds(X + r3 * (X_best - X), lb, ub)

VARIANTS = {
    'QSO-Full':       dict(theta_min=0.3, theta_max=0.7, tau=10,  use_centroid=True),
    'QSO-Static':     dict(theta_min=0.5, theta_max=0.5, tau=10,  use_centroid=True),
    'QSO-NoReset':    dict(theta_min=0.3, theta_max=0.7, tau=ITERS, use_centroid=True),
    'QSO-NoCentroid': dict(theta_min=0.3, theta_max=0.7, tau=10,  use_centroid=False),
}

def run_variant(func, lb, ub, dim, seed, cfg):
    """Runs qso() unmodified, or with exploitation_phase substituted for NoCentroid."""
    kw = dict(pop_size=POP, max_iter=ITERS, theta_min=cfg['theta_min'],
              theta_max=cfg['theta_max'], lambda_=0.05, tau=cfg['tau'],
              alpha=1.25, seed=seed)
    if cfg['use_centroid']:
        return qso(func, lb, ub, dim, **kw)[0]
    g = {**qso.__globals__, 'exploitation_phase': exploitation_phase_no_centroid}
    qso_nc = types.FunctionType(qso.__code__, g, qso.__name__,
                                qso.__defaults__, qso.__closure__)
    return qso_nc(func, lb, ub, dim, **kw)[0]

print('variants:', list(VARIANTS))

In [ ]:
# === 4. Official CEC 2017 functions ===
from opfunu.cec_based import cec2017

_FC = {}
def make_func(fid):
    if fid not in _FC:
        _FC[fid] = getattr(cec2017, f'F{fid}2017')(ndim=DIM)
    return _FC[fid]

f = make_func(1)
print('F1 at origin :', f.evaluate(np.zeros(DIM)))
print('bounds       :', f.lb[0], f.ub[0])
print()
print('These are the official competition shift vectors and rotation matrices.')
print('Locally generated substitutes give different functions and different conclusions')
print('— see Section 5.2 of the manuscript.')

In [ ]:
# === 5. Run (resumable) ===

def job(task):
    variant, fid, seed = task
    fn = make_func(fid)
    val = run_variant(fn.evaluate, float(fn.lb[0]), float(fn.ub[0]), DIM, seed, VARIANTS[variant])
    return variant, fid, seed, float(val)

results = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
tasks = [(v, f, s) for f in FUNCS for v in VARIANTS for s in SEEDS
         if len(results.get(f'{v}|{f}', [])) < len(SEEDS)]
print(f'{len(tasks)} runs remaining of {len(VARIANTS)*len(FUNCS)*len(SEEDS)}')

t0 = time.time()
buf = {}
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    for i, (v, fid, seed, val) in enumerate(ex.map(job, tasks, chunksize=4)):
        buf.setdefault(f'{v}|{fid}', []).append(val)
        if (i + 1) % 120 == 0:
            for k, vals in buf.items():
                results.setdefault(k, []).extend(vals)
            buf = {}
            json.dump(results, open(RESULTS, 'w'))
            print(f'  {i+1}/{len(tasks)}  [{(time.time()-t0)/60:.1f} min]', flush=True)
for k, vals in buf.items():
    results.setdefault(k, []).extend(vals)
json.dump(results, open(RESULTS, 'w'))
print(f'DONE — {sum(len(v) for v in results.values())} runs in {(time.time()-t0)/60:.1f} min')
print('saved:', RESULTS)

In [ ]:
# === 6. Table 7 ===
V = list(VARIANTS)
means = {v: {f: np.mean(results[f'{v}|{f}']) for f in FUNCS} for v in V}
stds  = {v: {f: np.std(results[f'{v}|{f}'], ddof=1) for f in FUNCS} for v in V}

ranks = {v: [] for v in V}
rows = []
for f in FUNCS:
    m = [means[v][f] for v in V]
    r = rankdata(m)
    for i, v in enumerate(V):
        ranks[v].append(r[i])
    rows.append([f'F{f}'] + [f'{means[v][f]:.3e} ({stds[v][f]:.2e}) R{r[i]:g}'
                             for i, v in enumerate(V)])
mfr = {v: float(np.mean(ranks[v])) for v in V}
rows.append(['MFR'] + [f'{mfr[v]:.3f}' for v in V])

print('TABLE 7 — component ablation, official CEC 2017, 30D, 30 runs')
print(pd.DataFrame(rows, columns=['F'] + [v.replace('QSO-', '') for v in V]).to_string(index=False))

print('\nW/D/L against QSO-Full (Wilcoxon per function, Holm-corrected across 10 functions):')
for v in V[1:]:
    ps = []
    for f in FUNCS:
        a = np.array(results[f'QSO-Full|{f}']); b = np.array(results[f'{v}|{f}'])
        if np.allclose(a, b):
            ps.append((f, 1.0, 0)); continue
        ps.append((f, wilcoxon(a, b)[1], np.sign(np.mean(b) - np.mean(a))))
    ps.sort(key=lambda x: x[1]); m = len(ps); w = dd = l = 0
    for i, (f, p, sgn) in enumerate(ps):
        padj = min(1.0, p * (m - i))
        if padj < 0.05 and sgn > 0: w += 1
        elif padj < 0.05 and sgn < 0: l += 1
        else: dd += 1
    ident = [f for f in FUNCS if np.allclose(results[f'QSO-Full|{f}'], results[f'{v}|{f}'])]
    print(f'  {v:<16} {w}W / {dd}D / {l}L' +
          (f'   (identical to QSO-Full on {len(ident)} functions: {ident})' if ident else ''))

## Expected output

The manuscript reports mean Friedman ranks of **QSO-Full 2.000, QSO-Static 2.400,
QSO-NoReset 1.800, QSO-NoCentroid 3.800**, with W/D/L against QSO-Full of 2W/6D/2L
(Static), 0W/10D/0L (NoReset) and 6W/4D/0L (NoCentroid). QSO-NoReset should reproduce
QSO-Full exactly on five of the ten functions, which is the evidence that the signal-decay
mechanism is inert at λ = 0.05.

Because seeds are fixed and the algorithm is deterministic given a seed, these figures
should reproduce exactly on any machine with `opfunu==1.0.1`.

## Note on the NoCentroid result

Removing the centroid term degrades performance on the official rotated functions but
appeared harmless on the earlier unrotated implementation. That reversal is reported in the
manuscript rather than resolved silently: it is a concrete instance of the argument in
Section 2.2 that benchmark implementation choices can determine conclusions.